# GraphRAG with Neo4j

This notebook shows the complete GraphRAG flow

**Documents → Chunking → Graph Extraction → Neo4j Storage → Graph Retrieval → LLM Answer**


In [1]:
!pip install --upgrade --quiet langchain langchain-community langchain-experimental langchain-groq langchain-neo4j neo4j


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.8/161.8 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 36.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

## 1. Connect to Neo4j

Add your own Neo4j Aura credentials below.


In [ ]:
from langchain_neo4j import Neo4jGraph

NEO4J_URI = "your_neo4j_connection_string"  # Replace with your actual Neo4j connection string
NEO4J_USERNAME = "your_username"
NEO4J_PASSWORD = "your_password"
NEO4J_DATABASE = "your_database"

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

print("Connected to Neo4j")


Connected to Neo4j


## 2. Create the LLM

The LLM will help us extract entities and relationships from the movie documents.


In [ ]:
from langchain_groq import ChatGroq

groq_api_key = "your_groq_api_key"

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    groq_api_key=groq_api_key,
    reasoning_effort="low",
    model_kwargs={
        "parallel_tool_calls": False
    }
)

## 3. Create a tiny movie dataset

Think of these as small documents coming from a website, PDF, database export, etc.


In [4]:
from langchain_core.documents import Document

movie_texts = [
    "Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.",
    "Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.",
    "Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008."
]

documents = [Document(page_content=text) for text in movie_texts]

documents


[Document(metadata={}, page_content='Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.'),
 Document(metadata={}, page_content='Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.'),
 Document(metadata={}, page_content='Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008.')]

## 4. Chunk the documents

In a real application, documents can be large. We split them into smaller chunks before sending them to the LLM.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

chunks = text_splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:", chunk.page_content)


Chunk 1: Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.
Chunk 2: Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.
Chunk 3: Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008.


## 5. Convert chunks into graph data

The LLM reads every chunk and extracts:
- **Nodes**: Person, Movie, Genre
- **Relationships**: DIRECTED, ACTED_IN, IN_GENRE


In [9]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Person", "Movie", "Genre"],
    allowed_relationships=["DIRECTED", "ACTED_IN", "IN_GENRE"]
)

graph_documents = llm_transformer.convert_to_graph_documents(chunks)


In [10]:
graph_documents

[GraphDocument(nodes=[Node(id='Christopher Nolan', type='Person', properties={}), Node(id='Leonardo Dicaprio', type='Person', properties={}), Node(id='Inception', type='Movie', properties={}), Node(id='Science Fiction', type='Genre', properties={})], relationships=[Relationship(source=Node(id='Christopher Nolan', type='Person', properties={}), target=Node(id='Inception', type='Movie', properties={}), type='DIRECTED', properties={}), Relationship(source=Node(id='Leonardo Dicaprio', type='Person', properties={}), target=Node(id='Inception', type='Movie', properties={}), type='ACTED_IN', properties={}), Relationship(source=Node(id='Inception', type='Movie', properties={}), target=Node(id='Science Fiction', type='Genre', properties={}), type='IN_GENRE', properties={})], source=Document(metadata={}, page_content='Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.')),
 GraphDocument(nodes=[Node(id='Christopher No

In [11]:
# Inspect extracted graph data
for i, graph_doc in enumerate(graph_documents):
    print(f"\n--- Graph Document {i+1} ---")
    print("Nodes:")
    for node in graph_doc.nodes:
        print(node)
    print("Relationships:")
    for rel in graph_doc.relationships:
        print(rel)



--- Graph Document 1 ---
Nodes:
id='Christopher Nolan' type='Person' properties={}
id='Leonardo Dicaprio' type='Person' properties={}
id='Inception' type='Movie' properties={}
id='Science Fiction' type='Genre' properties={}
Relationships:
source=Node(id='Christopher Nolan', type='Person', properties={}) target=Node(id='Inception', type='Movie', properties={}) type='DIRECTED' properties={}
source=Node(id='Leonardo Dicaprio', type='Person', properties={}) target=Node(id='Inception', type='Movie', properties={}) type='ACTED_IN' properties={}
source=Node(id='Inception', type='Movie', properties={}) target=Node(id='Science Fiction', type='Genre', properties={}) type='IN_GENRE' properties={}

--- Graph Document 2 ---
Nodes:
id='Christopher Nolan' type='Person' properties={}
id='Matthew Mcconaughey' type='Person' properties={}
id='Interstellar' type='Movie' properties={}
id='Science Fiction' type='Genre' properties={}
Relationships:
source=Node(id='Christopher Nolan', type='Person', properti

## 6. Store the graph in Neo4j

Now we persist the extracted nodes and relationships into Neo4j.


In [12]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

print("Graph data added to Neo4j")


Graph data added to Neo4j


## 7. Refresh and inspect the Neo4j schema

This helps us see what labels and relationships now exist in the graph.


In [13]:
graph.refresh_schema()
print(graph.schema)


Node properties:
Document {id: STRING, text: STRING}
Person {id: STRING}
Movie {id: STRING}
Genre {id: STRING}
Relationship properties:

The relationships:
(:Document)-[:MENTIONS]->(:Person)
(:Document)-[:MENTIONS]->(:Movie)
(:Document)-[:MENTIONS]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)


## 8. Retrieval: ask Neo4j for connected facts

This is the **R** in RAG. Instead of only finding similar text, GraphRAG can traverse relationships.


Find every movie that a director directed. Then, for each movie, find the actors who acted in it. Finally, return the director, movie, and all actors together.

In [14]:
result = graph.query("""
MATCH (director:Person)-[:DIRECTED]->(movie:Movie)
OPTIONAL MATCH (actor:Person)-[:ACTED_IN]->(movie)
RETURN director.id AS director, movie.id AS movie, collect(actor.id) AS actors
""")

result


[{'director': 'Christopher Nolan',
  'movie': 'Inception',
  'actors': ['Leonardo Dicaprio']},
 {'director': 'Christopher Nolan',
  'movie': 'Interstellar',
  'actors': ['Matthew Mcconaughey']},
 {'director': 'Christopher Nolan',
  'movie': 'The Dark Knight',
  'actors': ['Christian Bale']}]

## 9. Generation: let the LLM answer using the graph

`GraphCypherQAChain` does three simple things:
1. Reads the Neo4j schema
2. Converts the user question into Cypher
3. Executes Cypher and sends the retrieved facts to the LLM for the final answer


In [15]:
from langchain_neo4j import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True
)


In [19]:
response = chain.invoke({
    "query": "Which movies did Christopher Nolan direct?"
})

print(response["result"])




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {id: "Christopher Nolan"})-[:DIRECTED]->(m:Movie)
RETURN m
Full Context:
[{'m': {'id': 'Inception'}}, {'m': {'id': 'Interstellar'}}, {'m': {'id': 'The Dark Knight'}}]

> Finished chain.
Inception, Interstellar, The Dark Knight.


In [17]:
response = chain.invoke({
    "query": "Who acted in Inception?"
})

print(response["result"])




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m:Movie {id: 'Inception'})
RETURN p
Full Context:
[{'p': {'id': 'Leonardo Dicaprio'}}]

> Finished chain.
Leonardo DiCaprio acted in Inception.


In [20]:
response = chain.invoke({
    "query": "Which movie did Christopher Nolan direct in 2010 and who acted in it?"
})

print(response["result"])




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {id: "Christopher Nolan"})-[:DIRECTED]->(m:Movie)<-[:ACTED_IN]-(a:Person)
RETURN m.id AS movieId, a.id AS actorId
Full Context:
[{'movieId': 'Inception', 'actorId': 'Leonardo Dicaprio'}, {'movieId': 'Interstellar', 'actorId': 'Matthew Mcconaughey'}, {'movieId': 'The Dark Knight', 'actorId': 'Christian Bale'}]

> Finished chain.
Inception, starring Leonardo DiCaprio.


In [21]:
response = chain.invoke({
    "query": "Which movie did DSwithBappy direct?"
})

print(response["result"])




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {id: 'DSwithBappy'})-[:DIRECTED]->(m:Movie)
RETURN m
Full Context:
[]

> Finished chain.
I don't know the answer.


## Complete GraphRAG Flow

```text
Movie Documents
      ↓
Chunking
      ↓
LLM extracts entities + relationships
      ↓
Neo4j Knowledge Graph
      ↓
User Question
      ↓
Question → Cypher
      ↓
Neo4j retrieves connected facts
      ↓
Retrieved facts + question → LLM
      ↓
Final Answer
```

